In [0]:
-- create dim_product (flattened: aisle + department joined in, NULLs mapped to "Unknown")
CREATE OR REPLACE TABLE `ftw-week-06`.`03-mart`.dim_product AS
SELECT
    p.product_id,
    p.product_name,
    COALESCE(a.aisle, 'Unknown') AS aisle,
    COALESCE(d.department, 'Unknown') AS department
FROM `ftw-week-06`.`02-clean`.products_clean p
LEFT JOIN `ftw-week-06`.`02-clean`.aisles_clean a
    ON p.aisle_id = a.aisle_id
LEFT JOIN `ftw-week-06`.`02-clean`.departments_clean d
    ON p.department_id = d.department_id;


In [0]:
-- Create dim_order only if it does not already exist. (user_id folded in as an attribute, not its own dimension)
-- WHERE 1 = 0 creates the table structure without loading records.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`03-mart`.dim_order AS
SELECT
    order_id,
    user_id,
    eval_set,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order
FROM `ftw-week-06`.`02-clean`.orders_clean
WHERE 1 = 0;


-- Incrementally load new or updated orders into dim_order.
-- Existing order_id values are updated, while new orders are inserted.
MERGE INTO `ftw-week-06`.`03-mart`.dim_order AS target
USING `ftw-week-06`.`02-clean`.orders_clean AS source
ON target.order_id = source.order_id

-- Update an existing order if its attributes have changed.
WHEN MATCHED THEN UPDATE SET
    target.user_id = source.user_id,
    target.eval_set = source.eval_set,
    target.order_number = source.order_number,
    target.order_dow = source.order_dow,
    target.order_hour_of_day = source.order_hour_of_day,
    target.days_since_prior_order = source.days_since_prior_order

-- Insert a new order when the order_id does not yet exist.
WHEN NOT MATCHED THEN INSERT (
    order_id,
    user_id,
    eval_set,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order
)
VALUES (
    source.order_id,
    source.user_id,
    source.eval_set,
    source.order_number,
    source.order_dow,
    source.order_hour_of_day,
    source.days_since_prior_order
);

In [0]:
-- Create fact_order_products if it does not already exist. (one row per product within an order)
-- WHERE 1 = 0 creates the table structure without loading records.
CREATE TABLE IF NOT EXISTS `ftw-week-06`.`03-mart`.fact_order_products AS
SELECT
    order_id,
    product_id,
    add_to_cart_order,
    reordered
FROM `ftw-week-06`.`02-clean`.order_products_clean
WHERE 1 = 0;


-- Insert only order-product records that are not already in the fact table.
-- order_id + product_id serve as the composite key.
INSERT INTO `ftw-week-06`.`03-mart`.fact_order_products (
    order_id,
    product_id,
    add_to_cart_order,
    reordered
)
SELECT
    source.order_id,
    source.product_id,
    source.add_to_cart_order,
    source.reordered
FROM `ftw-week-06`.`02-clean`.order_products_clean AS source
WHERE NOT EXISTS (
    SELECT target.order_id, target.product_id
    FROM `ftw-week-06`.`03-mart`.fact_order_products AS target
    WHERE target.order_id = source.order_id
      AND target.product_id = source.product_id
);